# Chapter 3: An Overview of Long-Run Economic Growth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch03-an-overview-of-long-run-economic-growth.ipynb)

*Data Science and Macroeconomic Models: A Python Companion to Intermediate Macroeconomics* · Carlos Mendez

> **How to use this notebook.** In Colab, click *Runtime → Run all*. After that, every
> interactive figure has live sliders, buttons or menus: change one and the figure is drawn again
> on the spot. On the book website the same figures come with one precomputed slider or a few
> buttons. Each section ends with a short **Your turn** task, and you can try it in the empty code
> cell below the task.

## Learning goals

This chapter sets out the facts of economic growth and the tools we use to study them. By the end
of it you will be able to:

- describe how recent sustained growth is, and how unevenly it spread across countries;
- use the **constant growth rule** $y_t = y_0(1+\bar{g})^t$ and the **Rule of 70** for doubling times;
- read a **ratio scale**, on which constant growth is a straight line;
- compute an average annual growth rate between any two years;
- apply the rules for growth rates of products, ratios and powers;
- reshape, filter and summarize data tables with pandas, and draw line charts, scatter plots and step
  charts with Plotly Express.

## Figures in this chapter

The figures follow the order of the textbook chapter. The second column shows which textbook
figure each notebook figure replicates.

| Figure | Textbook figure | Type | What you can change |
|---|---|---|---|
| Figure 1 | 3.1 Economic growth over the very long run | data | the first year shown, linear or ratio scale |
| Figure 2 | 3.2 Per capita GDP in the United States | data | — (hover to read values) |
| Figure 3 | 3.3 Population over time | model | starting population, growth rate |
| Figure 4 | 3.4 Population over time, revisited | model | growth rate |
| Figure 5 | 3.5 U.S. per capita GDP on a ratio scale | data | the growth rate of the dashed line |
| Figure 6 | 3.6 Per capita GDP since 1870 | data | decade or annual data |
| Figure 7 | 3.7 Levels and growth rates of per capita GDP | data | the starting year (play button) |
| Figure 8 | 3.8 The world's population by per capita GDP | data | the year compared with 1960 |
| Figure 9 | 3.9 U.S. population, GDP and per capita GDP | data | the period |

The textbook's Table 3.1 on growth-rate rules is rebuilt inside Figure 9.

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                              # numbers and arrays
import pandas as pd                             # data tables
import plotly.express as px                     # quick charts from data tables
import plotly.graph_objects as go               # charts built piece by piece (model figures)
import plotly.io as pio
from plotly.subplots import make_subplots       # several panels in one figure
from ipywidgets import interact, FloatSlider, IntRangeSlider, ToggleButtons, Dropdown

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47", "#e0a33b", "#b04a5a"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch03/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each interactive figure also gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## The model in code

This chapter has no economic model yet, but it has three tools that we will use in every later
chapter. A **growth rate** is a percentage change: between year $t$ and year $t+1$ it is
$(y_{t+1} - y_t)/y_t$. If a variable grows at a constant rate $\bar{g}$ every year, starting from $y_0$, then
after $t$ years it equals

$$y_t = y_0\,(1+\bar{g})^t \qquad (3.7)$$

This is the **constant growth rule**. Turning it around gives the **average annual growth rate** between
year 0 and year $t$:

$$\bar{g} = \left(\frac{y_t}{y_0}\right)^{1/t} - 1 \qquad (3.9)$$

Finally, the **Rule of 70**: a variable growing at $g$ percent a year doubles in about $70/g$ years.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the
> *arguments*) and gives back a result with `return`. The text in triple quotes is a short
> description of what the function does. `**` means "to the power of", and `np.log` is the natural logarithm.

In [3]:
def constant_growth(y0, g, t):
    """Value after t years of constant growth at rate g: y_t = y0 (1 + g)^t   (equation 3.7)."""
    return y0 * (1 + g) ** t


def growth_rate(y0, yt, t):
    """Average annual growth rate between two values t years apart   (equation 3.9)."""
    return (yt / y0) ** (1 / t) - 1


def doubling_time(g):
    """Exact number of years it takes to double at growth rate g: solves 2 = (1 + g)^t."""
    return np.log(2) / np.log(1 + g)

Let us check the tools against the textbook.

> **Python note.** An *f-string* such as `f"{x:.1f}"` inserts the value of a variable into a piece
> of text; `:.1f` means "show one decimal" and `:.2%` shows a share as a percentage.

In [4]:
print(f"6 billion people growing at 2% for 100 years: {constant_growth(6, 0.02, 100):.1f} billion")
print(f"Growth from $4,200 (1870) to $76,000 (2022):  {growth_rate(4200, 76000, 152):.2%} a year")
print(f"Doubling time at 2%: Rule of 70 gives {70 / 2:.0f} years; the exact answer is {doubling_time(0.02):.1f}")

6 billion people growing at 2% for 100 years: 43.5 billion
Growth from $4,200 (1870) to $76,000 (2022):  1.92% a year
Doubling time at 2%: Rule of 70 gives 35 years; the exact answer is 35.0


## Figure 1: Economic growth over the very long run

**Goal.** See how recent sustained growth is.

The figure shows GDP per person over the last 2,000 years for six countries, measured as a multiple of
500 dollars (2011 prices), which is roughly the level at which people can barely survive.

In [5]:
longrun = pd.read_csv(DATA_URL + "fig3-1.csv")
longrun.tail()

,year,gdppc_x500_usa,gdppc_x500_jpn,gdppc_x500_gbr,gdppc_x500_arg,gdppc_x500_chn,gdppc_x500_gha
14,1990,73.96,59.90,52.38,20.51,5.96,3.39
15,2000,91.77,66.42,63.89,28.74,9.46,4.20
16,2010,98.53,70.02,69.51,37.96,19.32,5.89
17,2020,108.76,73.65,69.30,31.95,34.45,8.14
18,2022,116.97,76.54,76.81,36.58,38.48,8.48


The table is **wide**: one column per country. Plotly Express prefers **long** tables, with one row per
country and year.

> **Python note.** `.melt(id_vars="year")` turns the country columns into rows: the old column names go
> into a new column (`var_name`) and the numbers into another (`value_name`). `.map(NAMES)` then replaces
> each column name with a readable country name, using the dictionary `NAMES`. `.dropna()` removes the
> few years without an estimate, so that the lines join their neighbors.

In [6]:
NAMES = {"gdppc_x500_usa": "United States", "gdppc_x500_jpn": "Japan", "gdppc_x500_gbr": "United Kingdom",
         "gdppc_x500_arg": "Argentina", "gdppc_x500_chn": "China", "gdppc_x500_gha": "Ghana"}
longrun = longrun.melt(id_vars="year", var_name="series", value_name="gdp").dropna()
longrun["country"] = longrun.series.map(NAMES)
longrun.head()

,year,series,gdp,country
0,1,gdppc_x500_usa,1.70,United States
1,1000,gdppc_x500_usa,1.70,United States
2,1500,gdppc_x500_usa,1.70,United States
3,1600,gdppc_x500_usa,1.70,United States
5,1820,gdppc_x500_usa,5.35,United States


> **Python note.** `px.line` draws one line per value of `color=`; `markers=True` adds a dot for each
> estimate, and `log_y=True` gives a ratio scale (Figure 4 explains it). `interact` calls the function and
> draws the figure again every time you change a control: a `Dropdown` is a menu of choices, and
> `ToggleButtons` is a row of buttons. The `;` at the end only hides an extra line of text.

In [7]:
def very_long_run(start=1, scale="linear"):
    """Figure 3.1: GDP per person (multiples of $500) from the year `start` to 2022."""
    chosen = longrun[longrun.year >= start]
    uk = chosen[chosen.country == "United Kingdom"]
    factor = uk.gdp.iloc[-1] / uk.gdp.iloc[0]
    fig = px.line(chosen, x="year", y="gdp", color="country", markers=True, log_y=(scale == "ratio"),
                  labels={"year": "Year", "gdp": "GDP per person (multiple of 500 dollars)", "country": "Country"},
                  title=f"U.K. GDP per person in 2022 was {factor:.0f} times its level in the year {uk.year.iloc[0]}")
    if scale == "ratio":
        fig.update_yaxes(tickvals=[1, 2, 5, 10, 20, 50, 100])
    return fig

In [8]:
interact(very_long_run,
         start=Dropdown(options=[1, 1500, 1700, 1820, 1900, 1950], value=1, description="first year", style=WIDE),
         scale=ToggleButtons(options=["linear", "ratio"], description="vertical scale"));

In [9]:
# @title Website version: Figure 1 with buttons for the vertical scale and a range slider under the chart
fig = very_long_run()
fig.update_xaxes(rangeslider_visible=True)
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0.01, y=0.98, xanchor="left", yanchor="top", buttons=[
    dict(label="linear scale", method="relayout", args=[{"yaxis.type": "linear", "yaxis.autorange": True,
                                                         "yaxis.tickvals": None}]),
    dict(label="ratio scale", method="relayout", args=[{"yaxis.type": "log", "yaxis.autorange": True,
                                                        "yaxis.tickvals": [1, 2, 5, 10, 20, 50, 100]}])])])
fig

**What to notice.**

- For almost all of history, GDP per person stayed between 1 and 3 times the subsistence level, in every
  country (only the United Kingdom had pulled a little higher by 1500–1700). Then, in the last two or three
  centuries, it shot up: the curve looks like a hockey stick.
- Growth started at different times: first in the United Kingdom and the United States, later in Japan and
  Argentina, and only in the last few decades in China. Ghana's rise is much smaller.
- Choose a first year of 1820 or 1900 to stretch out the modern era, and switch to the ratio scale to
  separate the lines.

### Key numbers: the Great Divergence

In [10]:
latest = longrun[longrun.year == 2022].set_index("country").gdp
print((latest / latest["United States"]).round(2))

country
United States     1.00
Japan             0.65
United Kingdom    0.66
Argentina         0.31
China             0.33
Ghana             0.07
Name: gdp, dtype: float64


In 2022, Japan and the United Kingdom had about 2/3 of U.S. GDP per person, Argentina and China about
1/3, and Ghana about 1/14. Before 1700, incomes across countries probably differed by a factor of 2 or 3
at most. This widening of the gaps is called **the Great Divergence**.

**Your turn.** By what factor did GDP per person in Ghana rise between the year 1 and 2022? Compare it with
the factor for the United Kingdom in the title of the chart.

<details><summary>Hint</summary>

Pick out Ghana's rows with `longrun[longrun.country == "Ghana"]` and divide the last value of `gdp` by the
first one, as the function `very_long_run` does for the United Kingdom.

</details>

In [11]:
# Your turn: try it here

## Figure 2: Per capita GDP in the United States

**Goal.** Look at modern growth in one country, year by year.

In [12]:
us = pd.read_csv(DATA_URL + "fig3-2.csv")
factor = us.gdppc.iloc[-1] / us.gdppc.iloc[0]
fig = px.line(us, x="year", y="gdppc",
              labels={"year": "Year", "gdppc": "GDP per person (2022 dollars)"},
              title=f"U.S. GDP per person grew about {factor:.0f}-fold from 1870 to 2025")
fig.update_traces(line=dict(color=BLUE, width=3))
fig.show()

**What to notice.**

- The curve gets steeper over time: each year, income rises by a larger *amount* than the year before.
- The *percentage* change, however, stays roughly the same, around 2 percent a year. The change in income is
  roughly proportional to its level, $y_{t+1} - y_t \approx \bar{g}\,y_t$. That is exactly what a constant growth rate means.

> **Python note.** `us.set_index("year")` uses the years as row labels, so that `gdp[1870]` gives the value
> for 1870.

In [13]:
gdp = us.set_index("year").gdppc
print(f"1870: ${gdp[1870]:,}   2022: ${gdp[2022]:,}   ratio: {gdp[2022] / gdp[1870]:.1f}")
print(f"1970: ${gdp[1970]:,}   2000: ${gdp[2000]:,}   ratio: {gdp[2000] / gdp[1970]:.1f}")

1870: $4,634   2022: $77,942   ratio: 16.8
1970: $30,595   2000: $58,913   ratio: 1.9


GDP per person rose almost 17-fold between 1870 and 2022. Our 1870 value is about 10 percent higher than
the textbook's \$4,200, because we use the newer Maddison Project estimates for the years before 1929. In
the 30 years from 1970 to 2000, income almost doubled.

**Your turn.** Find the year in which U.S. GDP per person first passed \$40,000, and the year it first
passed \$80,000. How many years did it take to double?

<details><summary>Hint</summary>

`us[us.gdppc > 40000].head(1)` shows the first row above \$40,000. Do the same for \$80,000.

</details>

In [14]:
# Your turn: try it here

## Figure 3: Population over time

**Goal.** See what constant growth looks like when we plot it.

Suppose the world population starts at $L_0 = 6$ billion and grows at a constant rate $\bar{n} = 2\%$ a year.
Each year the population is multiplied by $(1+\bar{n})$:

$$L_{t+1} = L_t\,(1+\bar{n}) \qquad (3.2)$$

Applying this rule year after year gives $L_t = L_0(1+\bar{n})^t$ (3.6). The cell below checks this with a
loop: it multiplies by $1.02$ one hundred times and compares the result with the formula.

> **Python note.** A `for` loop repeats the indented lines once for each value. `range(100)` gives the
> numbers 0, 1, …, 99, so the loop runs exactly 100 times.

In [15]:
L = 6.0
for year in range(100):
    L = L * (1 + 0.02)                     # equation (3.2), applied once per year
print(f"Year by year:  L_100 = {L:.2f} billion")
print(f"Formula (3.6): L_100 = {constant_growth(6.0, 0.02, 100):.2f} billion")

Year by year:  L_100 = 43.47 billion
Formula (3.6): L_100 = 43.47 billion


> **Python note.** `np.arange(0, 101)` gives the whole numbers 0, 1, …, 100. `go.Figure()` starts an empty
> chart, and `add_trace` adds a curve to it. In the widget cell, a `FloatSlider` chooses a number between `min`
> and `max` in steps of `step`; `readout_format=".3f"` shows it with three decimals.

In [16]:
def population_path(L0=6.0, n=0.02):
    """Figure 3.3: population L_t = L0 (1 + n)^t over 100 years."""
    t = np.arange(0, 101)
    L = constant_growth(L0, n, t)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=L, name="Population", line=dict(color=BLUE, width=4)))
    fig.update_layout(title=f"L0 = {L0:.0f} billion growing at {100 * n:.1f}% a year: after 100 years, L = {L[-1]:.1f} billion",
                      xaxis=dict(title="Year", range=[0, 102], tickvals=[20, 40, 60, 80, 100]),
                      yaxis=dict(title="Population L (billions)", range=[0, 100]))
    return fig

In [17]:
interact(population_path,
         L0=FloatSlider(value=6, min=1, max=10, step=1, description="starting population L0", style=WIDE),
         n=FloatSlider(value=0.02, min=0.0, max=0.04, step=0.005, description="growth rate n", style=WIDE,
                       readout_format=".3f"));

In [18]:
# @title Website version: Figure 3 with a precomputed slider for the growth rate
slider_figure(population_path, "n", [0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04], start=0.02,
              label="growth rate n")

**What to notice.**

- At 2 percent a year, 6 billion people become 43.5 billion after a century, more than seven times as many.
  (This is only an illustration: demographers expect population growth to fall toward zero.)
- The curve bends upward, just like U.S. GDP per person in Figure 2. A constant growth rate adds a larger
  amount each year, because it applies to a larger base.
- From this picture alone, it is hard to tell whether the growth rate is constant. Figure 4 fixes that.

**Your turn.** Find the growth rate at which the population reaches about 18 billion, three times its
starting level, after 100 years. Then check your answer with `constant_growth`.

<details><summary>Hint</summary>

You need $(1+\bar{n})^{100} = 3$. Try the slider first; `growth_rate(6, 18, 100)` gives the exact answer.

</details>

In [19]:
# Your turn: try it here

## Figure 4: Population over time, revisited

**Goal.** Learn to read a **ratio scale**, on which constant growth is a straight line.

At 2 percent a year, the population doubles about every 35 years (Rule of 70: $70/2 = 35$). It goes from 6 to
12, 24 and 48 billion. On a **ratio scale**, the vertical axis is labeled 6, 12, 24, 48: each step doubles
the one before. Equal distances then mean equal *percentage* changes, so a constant growth rate draws a
straight line.

> **Python note.** `make_subplots(rows=1, cols=2, ...)` makes a figure with two panels side by side, and
> `row=1, col=2` sends a trace to the second one. `np.arange(0, 106, T)` gives 0, T, 2T, … up to 105. On a
> ratio-scale axis, the `range` is given in powers of ten: `np.log10(150)`.

In [20]:
def population_two_scales(n=0.02):
    """Figure 3.4: the same population path on (a) a standard scale and (b) a ratio scale."""
    T = doubling_time(n)
    t = np.linspace(0, 105, 200)
    dots = np.arange(0, 106, T)                                   # every doubling time
    L, L_dots = constant_growth(6, n, t), constant_growth(6, n, dots)
    fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) On a standard scale ...", "(b) and a ratio scale"))
    fig.add_trace(go.Scatter(x=t, y=L, name="Standard scale", line=dict(color=BLUE, width=4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=L, name="Ratio scale", line=dict(color=GREEN, width=4)), row=1, col=2)
    for col in [1, 2]:
        fig.add_trace(go.Scatter(x=dots, y=L_dots, name="Each doubling", mode="markers", showlegend=(col == 1),
                                 marker=dict(color="black", size=10)), row=1, col=col)
    fig.update_xaxes(title_text="Year", range=[-3, 108], tickvals=[25, 50, 75, 100])   # no label in the bottom-left corner
    fig.update_yaxes(title_text="Population L (billions)", range=[0, max(52, 1.05 * L[-1])], row=1, col=1)
    fig.update_yaxes(type="log", range=[np.log10(4.5), np.log10(150)], tickvals=[6, 12, 24, 48, 96], row=1, col=2)
    fig.update_layout(title=f"n = {100 * n:.1f}%: doubles every {T:.1f} years (Rule of 70: {70 / (100 * n):.1f})")
    return fig

In [21]:
interact(population_two_scales,
         n=FloatSlider(value=0.02, min=0.01, max=0.03, step=0.005, description="growth rate n", style=WIDE,
                       readout_format=".3f"));

In [22]:
# @title Website version: Figure 4 with a precomputed slider for the growth rate
slider_figure(population_two_scales, "n", [0.01, 0.015, 0.02, 0.025, 0.03], start=0.02, label="growth rate n")

**What to notice.**

- In panel (a), the dots are equally spaced in time (one every 35 years), but the jumps between them get
  bigger. In panel (b), the dots are equally spaced in both directions, and the curve is a straight line.
- A faster growth rate makes the line in panel (b) steeper. On a ratio scale, **the slope is the growth rate**.
- The Rule of 70 is very close to the exact doubling time for growth rates of a few percent.

**Your turn.** The textbook says that at 5 percent a year income doubles every 14 years. Use
`doubling_time` to compute the exact answer, and compare the two for a growth rate of 10 percent too.

<details><summary>Hint</summary>

`doubling_time(0.05)` and `doubling_time(0.10)`. The Rule of 70 gives $70/5$ and $70/10$. It becomes less
accurate as the growth rate rises.

</details>

In [23]:
# Your turn: try it here

## Figure 5: U.S. per capita GDP on a ratio scale

**Goal.** Use the ratio scale to read the growth rate of real data.

Here are the data of Figure 2 again, now on a ratio scale, with a dashed line that grows at a constant rate.
The line's level is chosen to fit the data as closely as possible (by least squares, in logarithms).

> **Python note.** `np.log(gdp)` takes the logarithm of every value at once, and `.mean()` averages
> them. On a ratio scale, a constant growth line is a straight line in logarithms, which is why we fit it there.

In [24]:
def ratio_scale_chart(g=0.02):
    """Figure 3.5: U.S. GDP per person on a ratio scale, with a constant-growth line at rate g."""
    years = us.year - 1870
    level = (np.log(us.gdppc) - years * np.log(1 + g)).mean()     # best-fitting starting level
    line = np.exp(level) * (1 + g) ** years
    fig = px.line(us, x="year", y="gdppc", log_y=True,
                  labels={"year": "Year", "gdppc": "GDP per person (2022 dollars, ratio scale)"})
    fig.update_traces(line=dict(color=BLUE, width=3), name="U.S. data", showlegend=True)
    fig.add_trace(go.Scatter(x=us.year, y=line, name=f"Constant growth at {100 * g:.2f}%",
                             line=dict(color=GREEN, width=3, dash="dash")))
    fig.update_yaxes(tickvals=[4000, 8000, 16000, 32000, 64000, 128000], range=[np.log10(3000), np.log10(130000)])
    fig.update_layout(title=f"Dashed line: {100 * g:.2f}% a year, doubling every {doubling_time(g):.0f} years")
    return fig

In [25]:
interact(ratio_scale_chart,
         g=FloatSlider(value=0.02, min=0.01, max=0.03, step=0.0025, description="growth rate of the line g",
                       style=WIDE, readout_format=".4f"));

In [26]:
# @title Website version: Figure 5 with a precomputed slider for the growth rate of the dashed line
slider_figure(ratio_scale_chart, "g", [0.01, 0.015, 0.0175, 0.02, 0.0225, 0.025, 0.03], start=0.02,
              label="growth rate of the line g")

**What to notice.**

- The data stay close to a straight line with a slope of about 2 percent a year for more than 150 years.
  This is one of the most remarkable facts in macroeconomics.
- Look closely: before 1929 the data rise a little more slowly than the line, and from 1950 to 2005 a
  little faster. The Great Depression (early 1930s) and World War II (early 1940s) are the big wiggles.
- Move the slider away from 2 percent: the line can no longer follow the data at both ends.

### Key numbers: calculating growth rates

The textbook applies equation (3.9) to \$4,200 in 1870 and \$76,000 in 2022. We do the same with our
data, and for two shorter periods.

In [27]:
print(f"Textbook numbers, 1870-2022: {growth_rate(4200, 76000, 152):.2%}")
print(f"Our data, 1870-2025:         {growth_rate(gdp[1870], gdp[2025], 155):.2%}")
print(f"Our data, 1870-1929:         {growth_rate(gdp[1870], gdp[1929], 59):.2%}")
print(f"Our data, 1950-2005:         {growth_rate(gdp[1950], gdp[2005], 55):.2%}")

Textbook numbers, 1870-2022: 1.92%
Our data, 1870-2025:         1.88%
Our data, 1870-1929:         1.56%
Our data, 1950-2005:         2.21%


**Your turn.** Compute the average growth rate of U.S. GDP per person from the year you were born to 2025.
Is it higher or lower than the long-run average?

<details><summary>Hint</summary>

Use `growth_rate(gdp[YEAR], gdp[2025], 2025 - YEAR)` with your birth year in place of `YEAR`.

</details>

In [28]:
# Your turn: try it here

## Figure 6: Per capita GDP since 1870

**Goal.** Compare the growth paths of several countries on a ratio scale.

The textbook plots one value per decade, to smooth out short-run ups and downs. We also have every
available year, from a wider data file, so you can see what the smoothing hides. We add India, which
the textbook discusses but does not draw.

In [29]:
decades = pd.read_csv(DATA_URL + "fig3-6.csv").melt(id_vars="year", var_name="series", value_name="gdppc")
decades["country"] = decades.series.map({"gdppc_usa": "United States", "gdppc_gbr": "United Kingdom",
                                         "gdppc_jpn": "Japan", "gdppc_arg": "Argentina", "gdppc_zaf": "South Africa",
                                         "gdppc_chn": "China", "gdppc_ind": "India"})
annual = pd.read_csv(DATA_URL + "maddison_panel.csv")
levels = decades.pivot(index="year", columns="country", values="gdppc")
japan = levels["Japan"] / levels["United States"]          # Japan relative to the U.S., by year
annual.tail()

,iso3,country,year,gdppc
960,ZAF,South Africa,2018,15319
961,ZAF,South Africa,2019,15147
962,ZAF,South Africa,2020,13984
963,ZAF,South Africa,2021,14544
964,ZAF,South Africa,2022,14728


> **Python note.** `category_orders` fixes the order of the countries in the legend. `.pivot(...)` turns the
> long table into a wide one, with one row per year and one column per country.

In [30]:
def countries_since_1870(frequency="every decade"):
    """Figure 3.6: GDP per person in seven countries on a ratio scale (2022 dollars)."""
    if frequency == "every decade":
        chosen = decades
    else:
        chosen = annual
    fig = px.line(chosen, x="year", y="gdppc", color="country", log_y=True,
                  category_orders={"country": ["United States", "United Kingdom", "Japan", "Argentina",
                                               "South Africa", "China", "India"]},
                  labels={"year": "Year", "gdppc": "GDP per person (2022 dollars, ratio scale)", "country": "Country"},
                  title=f"Japan went from {japan.iloc[0]:.0%} of U.S. GDP per person in 1870 to {japan.iloc[-1]:.0%} in 2022")
    fig.update_yaxes(tickvals=[1000, 2000, 4000, 8000, 16000, 32000, 64000], range=[np.log10(700), np.log10(90000)])
    return fig

In [31]:
interact(countries_since_1870,
         frequency=ToggleButtons(options=["every decade", "every year"], description="data"));

In [32]:
# @title Website version: Figure 6 with a precomputed slider for decade or annual data
slider_figure(countries_since_1870, "frequency", ["every decade", "every year"], start="every decade", label="data")

**What to notice.**

- The United Kingdom was the richest country in 1870, but grew more slowly than the United States and fell
  behind it between 1890 and 1900. Since 1950 the two lines are roughly parallel: they grow at about the same rate.
- Japan grew very fast after World War II and caught up with the United Kingdom. This catch-up is called
  **convergence**.
- Argentina grew well until about 1980 and slowly since. China and India took off only in the last decades.
- Click a country in the legend to hide it; double-click to show it alone.

### Key numbers: growth spurts and relative incomes

> **Python note.** `.pivot(index="year", columns="country", values="gdppc")` turns the long table back into
> a wide one, with one column per country, so that `wide.loc[1990, "Japan"]` picks a single number.

In [33]:
wide = annual.pivot(index="year", columns="country", values="gdppc")
print(f"Japan, 1950-1990: {growth_rate(wide.loc[1950, 'Japan'], wide.loc[1990, 'Japan'], 40):.1%} a year")
print(f"China, 1978-2022: {growth_rate(wide.loc[1978, 'China'], wide.loc[2022, 'China'], 44):.1%} a year")
print(f"India, 1990-2019: {growth_rate(wide.loc[1990, 'India'], wide.loc[2019, 'India'], 29):.1%} a year")
print("Relative to the United States in 2019:")
print((wide.loc[2019] / wide.loc[2019, "United States"]).round(2))

Japan, 1950-1990: 5.9% a year
China, 1978-2022: 5.6% a year
India, 1990-2019: 4.4% a year
Relative to the United States in 2019:
country
Argentina         0.32
China             0.30
India             0.13
Japan             0.68
South Africa      0.21
United Kingdom    0.69
United States     1.00
Name: 2019, dtype: float64


Japan grew by almost 6 percent a year from 1950 to 1990, as the textbook says. In these data China grew by
about 5.6 percent a year after 1978 and India by about 4.4 percent after 1990, a bit less than the textbook
reports: the Maddison Project has revised its estimates for both countries. In 2019, China had about 30
percent of U.S. GDP per person and India about 13 percent. On a ratio scale, a gap that *looks* small can be
large: check the numbers on the vertical axis.

**Your turn.** In which decade did Japan's GDP per person pass that of the United Kingdom? Find it in the
chart, then check it in the `wide` table.

<details><summary>Hint</summary>

`wide[wide["Japan"] > wide["United Kingdom"]].head(1)` shows the first year in which Japan was ahead.

</details>

In [34]:
# Your turn: try it here

## Figure 7: Levels and growth rates of per capita GDP

**Goal.** Look at growth in more than 100 countries at once.

We load a wider Penn World Table file with GDP per person and population for every country and year, and
keep the 111 countries with data since 1960, the textbook's sample.

In [35]:
pwt = pd.read_csv(DATA_URL + "pwt_income.csv")
sample = pwt[pwt.in_sample]
sample[sample.year == 2023].sort_values("gdp_pc").head()

,iso3,country,year,gdp_pc,y_rel_us_2019,pop_millions,in_sample
10844,VEN,Venezuela (Bolivarian Republic of),2023,89.3,0.00130,28.301,True
677,BDI,Burundi,2023,924.3,0.01346,13.689,True
1732,CAF,Central African Republic,2023,1069.6,0.01557,5.152,True
2227,COD,D.R. of the Congo,2023,1415.8,0.02061,105.790,True
7111,MOZ,Mozambique,2023,1461.8,0.02128,33.635,True


Look at the first row. Venezuela's GDP per person in 2023 appears to be about 0.1 percent of the U.S. level,
which is impossible. During Venezuela's hyperinflation the price data behind this number broke down. **Always look
at your data before using it.** We drop Venezuela and continue with 110 countries.

In [36]:
sample = sample[sample.country != "Venezuela (Bolivarian Republic of)"]

Now we compute, for each country, its average annual growth rate from a starting year to 2023, and its
GDP per person in 2023 relative to the United States.

> **Python note.** `merge` joins two tables on a common column, like matching rows by ID in a
> spreadsheet. Each pass of the loop adds one table to the list `tables`, and `pd.concat(tables)` stacks
> them into one long table. `.map(SHORT)` looks each country up in the dictionary `SHORT`; countries that
> are not in it get `NaN`, which `.fillna("")` replaces with an empty text. That way only a few names are
> printed on the chart, and `POSITION` puts some of them beside their dot so that labels do not overlap.

In [37]:
SHORT = {"United States": "United States", "China": "China", "India": "India", "Republic of Korea": "South Korea",
         "Botswana": "Botswana", "Singapore": "Singapore", "Japan": "Japan", "Brazil": "Brazil",
         "D.R. of the Congo": "DR Congo", "Burundi": "Burundi", "Ethiopia": "Ethiopia"}
POSITION = {"United States": "bottom right", "India": "top left", "Botswana": "bottom right",
            "Singapore": "middle left", "Japan": "bottom left", "D.R. of the Congo": "bottom center"}
end = sample[sample.year == 2023][["iso3", "country", "gdp_pc"]]
end["level_2023"] = end.gdp_pc / end[end.country == "United States"].gdp_pc.iloc[0]
tables = []
for start in [1960, 1970, 1980, 1990, 2000, 2010]:
    first = sample[sample.year == start][["iso3", "gdp_pc"]]
    table = end.merge(first, on="iso3", suffixes=("_2023", "_start"))
    table["growth"] = 100 * growth_rate(table.gdp_pc_start, table.gdp_pc_2023, 2023 - start)
    table["start_year"] = start
    tables.append(table)
growth = pd.concat(tables)
growth["label"] = growth.country.map(SHORT).fillna("")
growth["position"] = growth.country.map(POSITION).fillna("top center")
growth.head()

,iso3,country,gdp_pc_2023,level_2023,gdp_pc_start,growth,start_year,label,position
0,ARG,Argentina,27453.1,0.367739,3616.4,3.269780,1960,,top center
1,AUS,Australia,62890.6,0.842431,17361.6,2.064086,1960,,top center
2,AUT,Austria,62979.0,0.843615,11247.5,2.772101,1960,,top center
3,BDI,Burundi,924.3,0.012381,792.5,0.244496,1960,Burundi,top center
4,BEL,Belgium,61771.6,0.827442,12510.1,2.567172,1960,,top center


> **Python note.** `animation_frame="start_year"` adds a play button and a slider that step through the
> starting years. `log_x=True` puts the horizontal axis on a ratio scale, and `ticktext` writes the tick
> values as fractions. Each country gets its own label position, taken from the first frame (1960).

In [38]:
FRACTIONS = [1 / 128, 1 / 64, 1 / 32, 1 / 16, 1 / 8, 1 / 4, 1 / 2, 1, 2]
FRACTION_NAMES = ["1/128", "1/64", "1/32", "1/16", "1/8", "1/4", "1/2", "1", "2"]
fig = px.scatter(growth, x="level_2023", y="growth", text="label", hover_name="country",
                 animation_frame="start_year", log_x=True, range_x=[1 / 100, 2.2], range_y=[-4, 9],
                 color_discrete_sequence=[GREEN],                # the color must be set here to last in every frame
                 labels={"level_2023": "Per capita GDP in 2023 (ratio scale, U.S. = 1)",
                         "growth": "Average annual growth to 2023 (percent)", "start_year": "Starting year"},
                 title=f"Average growth to 2023 ranges from {growth.growth.min():.0f} to "
                       f"{growth.growth.max():+.0f} percent a year across countries")
fig.update_traces(marker=dict(size=9), textfont_size=10, textposition=growth[growth.start_year == 1960].position)
fig.update_xaxes(tickvals=FRACTIONS, ticktext=FRACTION_NAMES)
fig.show()

**What to notice.**

- From 1960 to 2023, growth rates range from about −1 to +6 percent a year. South Korea, China, Singapore,
  Botswana and Taiwan grew fastest; the Democratic Republic of the Congo, the Central African Republic,
  Madagascar and Niger got poorer.
- Most countries lie between 1 and 3 percent. The cloud tilts upward, but partly by construction: the
  horizontal axis shows income *today*, and fast growth is what made a country rich today.
- Press play: for starting years 2000 and 2010 the tilt disappears, and many poor countries grew as fast as
  rich ones or faster.

### Key numbers: the power of compounding

In [39]:
from_1960 = growth[growth.start_year == 1960].set_index("country")
g_sgp = from_1960.growth["Singapore"] / 100
print(f"Singapore grew {g_sgp:.1%} a year: it doubles every {doubling_time(g_sgp):.0f} years")
print(f"Over 63 years its income rose by a factor of {(1 + g_sgp) ** 63:.0f}")
print(f"Madagascar: {from_1960.growth['Madagascar']:.1f}% a year; Nicaragua: {from_1960.growth['Nicaragua']:.1f}% a year")

Singapore grew 5.9% a year: it doubles every 12 years
Over 63 years its income rose by a factor of 36
Madagascar: -0.2% a year; Nicaragua: 0.5% a year


**Your turn.** Find the countries whose GDP per person *fell* between 1960 and 2023. How many are there?

<details><summary>Hint</summary>

`from_1960[from_1960.growth < 0]` keeps the countries with negative growth. `len(...)` counts them.

</details>

In [40]:
# Your turn: try it here

## Figure 8: The world's population by per capita GDP

**Goal.** Count people instead of countries.

In Figure 7 China and Luxembourg each count as one dot. Here every person counts equally. For a given year,
we sort countries from poorest to richest and add up their populations. The chart shows the percentage of
the world's population that lives in countries with GDP per person at or below each level. Income is
measured relative to the U.S. level **in 2019** for every year, so that the curves can be compared.

> **Python note.** `.cumsum()` gives running totals: the first value, then the sum of the first two, and so
> on. Dividing by the total and multiplying by 100 turns them into cumulative percentages. `.copy()` makes a
> separate table, so that adding a column does not touch `sample`. `line_shape="hv"` draws the line as steps,
> one for each country, and `add_vline` draws the dashed vertical line.

In [41]:
def cumulative_shares(year):
    """Countries sorted from poorest to richest in `year`, with the cumulative share of world population."""
    table = sample[sample.year == year].sort_values("y_rel_us_2019").copy()
    table["share"] = 100 * table.pop_millions.cumsum() / table.pop_millions.sum()
    return table


def population_distribution(year=2019):
    """Figure 3.8: cumulative share of world population by GDP per person (U.S. 2019 = 1), year vs 1960."""
    both = pd.concat([cumulative_shares(1960), cumulative_shares(year)])
    below = cumulative_shares(year)
    poor = below[below.y_rel_us_2019 < 0.05].pop_millions.sum() / below.pop_millions.sum()
    fig = px.line(both, x="y_rel_us_2019", y="share", color="year", line_shape="hv", hover_name="country",
                  log_x=True, range_x=[1 / 200, 2], range_y=[0, 102], color_discrete_sequence=[GREEN, BLUE],
                  labels={"y_rel_us_2019": "Per capita GDP (U.S. = 1 in 2019, ratio scale)",
                          "share": "Share of world population (percent)", "year": "Year"})
    fig.add_vline(x=0.05, line_dash="dash", line_color=GRAY)
    fig.update_xaxes(tickvals=FRACTIONS, ticktext=FRACTION_NAMES)
    fig.update_layout(title=f"{year}: {poor:.0%} of people live below 5% of the 2019 U.S. level (dashed line)")
    return fig

In [42]:
interact(population_distribution,
         year=Dropdown(options=[1970, 1980, 1990, 2000, 2010, 2019, 2023], value=2019, description="year", style=WIDE));

In [43]:
# @title Website version: Figure 8 with a precomputed slider for the year
slider_figure(population_distribution, "year", [1970, 1980, 1990, 2000, 2010, 2019, 2023], start=2019, label="year")

**What to notice.**

- The whole distribution moves to the right over time: most of the world's people are much richer than in 1960.
- The big vertical jumps are China and India, each with more than a billion people. Hover over them. Their
  fast growth since 1980 moved hundreds of millions of people across the dashed poverty line.
- Move through the years: the share of people below the dashed line falls slowly until 1990 and then very fast.

### Key numbers: fewer people in poverty

In [44]:
shares_1960, shares_2019 = cumulative_shares(1960), cumulative_shares(2019)
poor_1960 = shares_1960[shares_1960.y_rel_us_2019 < 0.05].pop_millions.sum() / shares_1960.pop_millions.sum()
poor_2019 = shares_2019[shares_2019.y_rel_us_2019 < 0.05].pop_millions.sum()
total_2019 = shares_2019.pop_millions.sum()
print(f"Share below the line: 1960 {poor_1960:.1%}, 2019 {poor_2019 / total_2019:.1%}")
print(f"People below the line in 2019:              {poor_2019 / 1000:.2f} billion")
print(f"... if the 1960 distribution had not changed: {poor_1960 * total_2019 / 1000:.2f} billion")

Share below the line: 1960 68.7%, 2019 8.5%
People below the line in 2019:              0.58 billion
... if the 1960 distribution had not changed: 4.70 billion


The textbook reports "2 out of 3" in 1960 and "less than 1 out of 12" in 2019. Our 110 countries give
about 69 and 8.5 percent. With the 1960 distribution, about 4.7 billion people would be below the line
today; in fact about 0.6 billion are.

**Your turn.** Repeat the poverty count with a higher line, 10 percent of the 2019 U.S. level. How many
people were below it in 1960 and in 2019?

<details><summary>Hint</summary>

Copy the cell above and replace `0.05` with `0.10` in the three places where it appears.

</details>

In [45]:
# Your turn: try it here

## Figure 9: U.S. population, GDP and per capita GDP

**Goal.** See the rules for growth rates at work.

Three rules make growth rates easy to work with. If $x$ grows at rate $g_x$ and $y$ at rate $g_y$, then

- the product $z = x \times y$ grows at $g_x + g_y$;
- the ratio $z = x / y$ grows at $g_x - g_y$;
- the power $z = x^a$ grows at $a \times g_x$.

Total GDP is GDP per person times population, so its growth rate is the *sum* of the two growth rates. To
see this, the chart divides each series by its value in the first year, so all three lines start at 1.

> **Python note.** `start, end = period` unpacks a pair of years into two variables. Dividing the table
> `usa.loc[start:end]` by the row `usa.loc[start]` divides every column by its own first value. An
> `IntRangeSlider` has two handles, so it chooses a whole period at once.

In [46]:
usa = pd.read_csv(DATA_URL + "fig3-9.csv").set_index("year")


def growth_rules_chart(period=(1870, 2025)):
    """Figure 3.9: total GDP, GDP per person and population, each relative to its first-year value."""
    start, end = period
    if end == start:                                              # both handles on one year: use one year
        end = min(start + 1, 2025)
        start = end - 1
    chosen = usa.loc[start:end] / usa.loc[start]                  # every series starts at 1
    chosen = chosen.reset_index().rename(columns={"gdp_billions": "Total GDP", "gdppc": "Per capita GDP",
                                                  "population_thousands": "Population"})
    rates = 100 * growth_rate(usa.loc[start], usa.loc[end], end - start)
    fig = px.line(chosen, x="year", y=["Total GDP", "Per capita GDP", "Population"], log_y=True,
                  color_discrete_sequence=["#e0a33b", GREEN, BLUE],
                  labels={"year": "Year", "value": "Relative to the first year (ratio scale)", "variable": "Series"})
    fig.update_yaxes(tickvals=[1, 2, 5, 10, 20, 50, 100, 200])
    fig.update_xaxes(range=[start - (end - start) / 20, end])   # a little room: no label in the bottom-left corner
    fig.update_layout(title=f"{start}–{end}: GDP {rates.gdp_billions:.2f}%  ≈  per capita {rates.gdppc:.2f}%"
                            f" + population {rates.population_thousands:.2f}%  a year")
    return fig

In [47]:
interact(growth_rules_chart,
         period=IntRangeSlider(value=(1870, 2025), min=1870, max=2025, step=1, description="period", style=WIDE));

In [48]:
# @title Website version: Figure 9 with a precomputed slider for the starting year
PERIODS = [(1870, 2025), (1900, 2025), (1929, 2025), (1950, 2025), (1980, 2025), (2000, 2025)]
fig = slider_figure(growth_rules_chart, "period", PERIODS, start=(1870, 2025), label="period")
for step, (start, end) in zip(fig.layout.sliders[0].steps, PERIODS):      # each period keeps its own x range
    step.args = [step.args[0], {**step.args[1], "xaxis.range": [start - (end - start) / 20, end]}]
fig

**What to notice.**

- From 1870 to 2025, total GDP grew 3.3 percent a year, per capita GDP 1.9 percent and population 1.4
  percent. The first number is (almost exactly) the sum of the other two.
- The sum is not exact because growth over a whole year compounds: $(1+g_Y) = (1+g_y)(1+g_L)$. For small rates
  the extra term $g_y \times g_L$ is tiny.
- Population growth has slowed to less than 1 percent a year since 1980, so growth in GDP per person now
  accounts for a larger part of GDP growth. Choose a later starting year to see it.

### Table 3.1: examples of growth-rate calculations

Suppose $x$ grows at 10 percent and $y$ at 3 percent a year. The rules give the growth rate of each
combination. The last column checks the rule by computing the actual one-year growth of $z$ with numbers.

In [49]:
gx, gy = 0.10, 0.03
x0, y0 = 1.0, 1.0
x1, y1 = x0 * (1 + gx), y0 * (1 + gy)                          # the values one year later
rows = [
    ["z = x × y", gx + gy, (x1 * y1) / (x0 * y0) - 1],
    ["z = x / y", gx - gy, (x1 / y1) / (x0 / y0) - 1],
    ["z = y / x", gy - gx, (y1 / x1) / (y0 / x0) - 1],
    ["z = x²", 2 * gx, x1 ** 2 / x0 ** 2 - 1],
    ["z = y^(1/2)", 0.5 * gy, y1 ** 0.5 / y0 ** 0.5 - 1],
    ["z = x^(1/2) y^(-1/4)", 0.5 * gx - 0.25 * gy, (x1 ** 0.5 * y1 ** -0.25) - 1],
]
pd.DataFrame(rows, columns=["Case", "Rule", "Exact one-year growth"]).round(4)

,Case,Rule,Exact one-year growth
0,z = x × y,0.1300,0.1330
1,z = x / y,0.0700,0.0680
2,z = y / x,-0.0700,-0.0636
3,z = x²,0.2000,0.2100
4,z = y^(1/2),0.0150,0.0149
5,z = x^(1/2) y^(-1/4),0.0425,0.0411


The rules are approximations, but very good ones when growth rates are small. (They hold exactly for
growth measured continuously, in logarithms.)

**Your turn.** The textbook's case study uses $Y = A K^{1/3} L^{2/3}$. Suppose $A$ grows at 1 percent, $K$ at 3
percent and $L$ at 1 percent a year. Use the rules to compute the growth rate of $Y$, then check it with
numbers as in the table.

<details><summary>Hint</summary>

The rules give $g_Y = g_A + \frac{1}{3}g_K + \frac{2}{3}g_L$. For the check, start from $A = K = L = 1$ and compute
$Y$ one year later.

</details>

In [50]:
# Your turn: try it here

## Key numbers

This table collects the main numbers of the chapter. Each one is computed with the functions and data above.

In [51]:
rows = [
    ["World population after 100 years at 2% (from 6 billion)", constant_growth(6, 0.02, 100)],
    ["Doubling time at 2%, exact (years)", doubling_time(0.02)],
    ["U.S. GDP per person, 2022 / 1870", gdp[2022] / gdp[1870]],
    ["U.S. growth of GDP per person, 1870-2025 (%)", 100 * growth_rate(gdp[1870], gdp[2025], 155)],
    ["Japan's growth, 1950-1990 (%)", 100 * growth_rate(wide.loc[1950, "Japan"], wide.loc[1990, "Japan"], 40)],
    ["Singapore's growth, 1960-2023 (%)", 100 * g_sgp],
    ["Share of people below 5% of U.S. 2019 GDP per person, 1960 (%)", 100 * poor_1960],
    ["Share of people below 5% of U.S. 2019 GDP per person, 2019 (%)", 100 * poor_2019 / total_2019],
]
pd.DataFrame(rows, columns=["Quantity", "Value"]).round(2)

,Quantity,Value
0,World population after 100 years at 2% (from 6...,43.47
1,"Doubling time at 2%, exact (years)",35.00
2,"U.S. GDP per person, 2022 / 1870",16.82
3,"U.S. growth of GDP per person, 1870-2025 (%)",1.88
4,"Japan's growth, 1950-1990 (%)",5.87
5,"Singapore's growth, 1960-2023 (%)",5.87
6,Share of people below 5% of U.S. 2019 GDP per ...,68.66
7,Share of people below 5% of U.S. 2019 GDP per ...,8.54


## Summary

- Sustained growth in living standards is very recent. It began about 200 years ago and spread unevenly, which
  produced the **Great Divergence** in incomes across countries.
- A **growth rate** is a percentage change. With constant growth, $y_t = y_0(1+\bar{g})^t$, and a variable doubles in
  about $70/g$ years (**Rule of 70**).
- On a **ratio scale**, constant growth is a straight line and the slope is the growth rate. U.S. GDP per person has
  grown at close to 2 percent a year for more than 150 years.
- Growth rates differ a lot across countries, and small differences compound into huge income gaps. Counting people
  rather than countries, poverty has fallen dramatically, mainly because of growth in China and India.
- Growth rates of products are sums, of ratios are differences, and of powers are multiples.
- The next chapter builds a first model to explain why some countries are so much richer than others.